In [1]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim

In [2]:
# load the datasets
train = pd.read_csv("fashion-mnist_train.csv")
test = pd.read_csv("fashion-mnist_test.csv")

print(train.shape)
print(test.shape)

(60000, 785)
(10000, 785)


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
X_train = train.drop('label' , axis=1)
y_train = train['label']
X_test = test.drop('label' , axis=1)
y_test = test['label']

In [5]:
# convert to tensors
X_train = torch.from_numpy(np.array(X_train)).to(dtype=torch.float32)
X_test = torch.from_numpy(np.array(X_test)).to(dtype=torch.float32)
y_train = torch.from_numpy(np.array(y_train)).to(dtype=torch.long)
y_test = torch.from_numpy(np.array(y_test)).to(dtype=torch.long)

In [6]:
# standardize the data from 0 to 1

X_train = X_train/255.
X_test = X_test/255.

In [7]:
# create dataset and dataloader
from torch.utils.data import Dataset , DataLoader
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features.reshape(-1,1,28,28)
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self,idx):
        return self.features[idx] , self.labels[idx]

In [8]:
train_dataset = CustomDataset(X_train , y_train)
test_dataset = CustomDataset(X_test , y_test)

In [9]:
train_loader = DataLoader(train_dataset , batch_size=512 , shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset , batch_size=512 , shuffle=True,pin_memory=True)

In [21]:
## Model Building
import torch.nn as nn
class MyModel(nn.Module):
    def __init__(self , no_of_channels):

        super().__init__()

        self.convolution = nn.Sequential(
            nn.Conv2d(no_of_channels , 32, kernel_size = 3, padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2 , stride=2),

            nn.Conv2d(32 , 64, kernel_size = 3, padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2 , stride=2)

        )


        self.classifier = nn.Sequential(
            nn.Flatten(), # flatten layer

            nn.Linear(64*7*7 , 128),
            nn.BatchNorm1d(128), # added batch normalization 
            nn.ReLU(),
            nn.Dropout(p=0.4), # added dropouts with prob = 0.4

            nn.Linear(128 , 64),
            nn.BatchNorm1d(64), 
            nn.ReLU(),
            nn.Dropout(p=0.4),
            
            nn.Linear(64,10)
        )

    # forward pass

    def forward(self,features):
        out = self.convolution(features)
        out = self.classifier(out)
        return out

In [22]:
# Parameters
learning_rate = 0.1
epochs=100

# create the model Object
model = MyModel(1)
model = model.to(device) # this saves my model in GPU

# create loss

criterion = nn.CrossEntropyLoss()

# optimizer

optimizer = optim.SGD(model.parameters() ,weight_decay= 1e-4, lr = learning_rate)

In [23]:
for batch_features, batch_labels in train_loader:
    print(batch_features.shape)
    print(batch_labels.shape)
    break

torch.Size([512, 1, 28, 28])
torch.Size([512])


In [24]:
# Let us now build the back propagation code
import time
start_time = time.time()
for epoch in range(epochs):
    total_loss = 0
    for batch_features,batch_labels in train_loader:
        # save them in GPU
        batch_features,batch_labels = batch_features.to(device), batch_labels.to(device)

        # forward pass

        outputs = model(batch_features)

        # calc loss

        loss = criterion(outputs , batch_labels) # this automatically converts logits to probs using softmax

        # remove the gradients
        optimizer.zero_grad()

        # back propagate
        loss.backward()

        # update the weights
        optimizer.step()

        # find the sum of loss in each epoch
        total_loss += loss
    print(f'Loss at epoch {epoch+1} ->{total_loss/len(train_loader)}')
print(time.time()-start_time)

Loss at epoch 1 ->0.7593557834625244
Loss at epoch 2 ->0.4406053125858307
Loss at epoch 3 ->0.3692096173763275
Loss at epoch 4 ->0.33595454692840576
Loss at epoch 5 ->0.30600059032440186
Loss at epoch 6 ->0.28556299209594727
Loss at epoch 7 ->0.2692805826663971
Loss at epoch 8 ->0.2543819844722748
Loss at epoch 9 ->0.24086043238639832
Loss at epoch 10 ->0.23180024325847626
Loss at epoch 11 ->0.21689069271087646
Loss at epoch 12 ->0.21115711331367493
Loss at epoch 13 ->0.20335857570171356
Loss at epoch 14 ->0.19526858627796173
Loss at epoch 15 ->0.1874985247850418
Loss at epoch 16 ->0.1783444881439209
Loss at epoch 17 ->0.17064635455608368
Loss at epoch 18 ->0.16849800944328308
Loss at epoch 19 ->0.1575334072113037
Loss at epoch 20 ->0.1521255522966385
Loss at epoch 21 ->0.1463831663131714
Loss at epoch 22 ->0.1413569152355194
Loss at epoch 23 ->0.13727423548698425
Loss at epoch 24 ->0.13149841129779816
Loss at epoch 25 ->0.12419843673706055
Loss at epoch 26 ->0.12287130206823349
Loss a

In [25]:
# Evaluation 
# set to eval mode
model.eval()
total = 0
correct = 0
for batch_features , batch_labels in test_loader:
    batch_features , batch_labels  = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # get the max values of outputs
    _,predicted = torch.max(outputs , 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy Score -> {correct/total}')



Accuracy Score -> 0.9257


In [26]:
# Evaluation of training data
# set to eval mode
model.eval()
total = 0
correct = 0
for batch_features , batch_labels in train_loader:
    batch_features , batch_labels  = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # get the max values of outputs
    _,predicted = torch.max(outputs , 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy Score -> {correct/total}')



Accuracy Score -> 0.99975


- We can see that our model is overfitting, to reduce overfitting we can perform a few things:
    - 1. Introduce dropouts after each layer.
    - 2. Use BatchNormalization in each layer to normalize the output data of every hidden layer
    - 3. Third is to use regularization (L2) using weight_decay